In [32]:
# 提取数据
import pandas as pd

data = pd.read_csv("train.csv.gz")
y = data.SalePrice
data_features = ['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'MasVnrArea', 'TotalBsmtSF', 'GrLivArea', 'MiscVal']
x = data[data_features]

print("All the data have been read.")

All the data have been read.


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# 为了确保模型的泛化能力，我们需要将数据划分为训练集和验证集
train_x, test_x, train_y, test_y = train_test_split(x, y, random_state=0)
model = RandomForestRegressor()
model.fit(train_x, train_y)
predicted_home_prices = model.predict(test_x)

# 计算平均绝对误差，同时初步检验模型效果
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(test_y, predicted_home_prices))

21024.081619221568


In [34]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [200, 300, 500, 1000],
    "max_depth": [10, 20, 30, 50],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", "log2"]
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,                         
    cv=5,                              
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 在训练集上找最佳参数
search.fit(x, y)

print("最佳参数:", search.best_params_)
print("交叉验证最佳 MAE:", -search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
最佳参数: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 50}
交叉验证最佳 MAE: 20241.76610601032


In [35]:
model = RandomForestRegressor(**search.best_params_, n_jobs=-1)
model.fit(x, y)

# 处理测试集数据
ans_data = pd.read_csv("test.csv.gz")
ans_ids = ans_data["Id"] 
ans_x = ans_data[data_features]
ans_predicted_price = model.predict(ans_x)

output = pd.DataFrame({
    "Id": ans_ids,
    "SalePrice": ans_predicted_price
})
output.to_csv("output.csv", index=False)
print(output.head())
print("Successfully Predicted the House Price!!")


     Id      SalePrice
0  1461  130490.080422
1  1462  169473.064525
2  1463  155988.008602
3  1464  174881.389158
4  1465  202893.178526
Successfully Predicted the House Price!!
